#### Inputs

In [2]:
# Bibliotecas
from brukeropus import read_opus
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import tempfile
import optuna
import boto3
import os

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Configurações
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Mudança no display
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Usa as credenciais já configuradas no seu ambiente AWS CLI
session = boto3.Session(region_name="us-east-1")
s3 = session.client("s3")

def avaliar_modelos_espectrais_optuna(
    df,
    target_col,
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=30,
    n_jobs=-1,
    verbose=0
):
    """
    Avalia múltiplos modelos espectrais usando Optuna.

    Variações testadas:
    1) StandardScaler -> PCA -> Modelo
    2) StandardScaler -> Modelo (PLS sem PCA)
    3) StandardScaler -> SelectKBest -> ElasticNet
    4) Modelo puro sem PCA/Scaler para boosting baseado em árvores

    Retorna:
    - resultados_df
    - best_models
    - split_data
    """

    # Preparação dos dados
    feature_cols = [c for c in df.columns if c.startswith(feature_prefix)]
    if len(feature_cols) == 0:
        raise ValueError(f"Nenhuma coluna encontrada com prefixo '{feature_prefix}'.")

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    resultados = []
    best_models = {}

    # Lista de modelos / variações
    nomes_modelos = [
        "Ridge",
        "PLS",
        "ElasticNet",
        "ElasticNet_SelectKBest",
        "Lasso",
        "XGBoost",
        "LightGBM",
        "MLP"
    ]

    max_pca = min(80, X_train.shape[1], X_train.shape[0] - 1)
    min_pca = min(10, max_pca)

    if min_pca < 2:
        raise ValueError("Poucas amostras/features para usar PCA com segurança.")

    max_kbest = min(300, X_train.shape[1])
    min_kbest = min(30, max_kbest)

    if min_kbest < 2:
        raise ValueError("Poucas features para usar SelectKBest com segurança.")

    def montar_pipeline(trial, nome_modelo):
        # Modelos com PCA
        if nome_modelo in [
            "Ridge", 
            "PLS", 
            "ElasticNet", 
            "Lasso",
            "XGBoost", 
            "LightGBM", 
            "MLP"
        ]:
            pca_n = trial.suggest_int("pca_n_components", min_pca, max_pca)

        if nome_modelo == "Ridge":
            alpha = trial.suggest_float("alpha", 1e-3, 1e3, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", Ridge(alpha=alpha))
            ])

        elif nome_modelo == "PLS":
            n_comp = trial.suggest_int("n_components", 2, min(30, pca_n))

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", PLSRegression(n_components=n_comp))
            ])

        elif nome_modelo == "ElasticNet":
            alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
            l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    max_iter=20000,
                    random_state=random_state
                ))
            ])

        elif nome_modelo == "ElasticNet_SelectKBest":
            k = trial.suggest_int("k_best", min_kbest, max_kbest)
            alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
            l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("select", SelectKBest(score_func=f_regression, k=k)),
                ("model", ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    max_iter=20000,
                    random_state=random_state
                ))
            ])

        elif nome_modelo == "Lasso":
            alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", Lasso(
                    alpha=alpha,
                    max_iter=20000,
                    random_state=random_state
                ))
            ])

        elif nome_modelo == "XGBoost":
            n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
            learning_rate = trial.suggest_float("learning_rate", 1e-2, 0.2, log=True)
            max_depth = trial.suggest_int("max_depth", 2, 8)
            subsample = trial.suggest_float("subsample", 0.6, 1.0)
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)
            reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10, log=True)
            reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", XGBRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=subsample,
                    colsample_bytree=colsample_bytree,
                    reg_alpha=reg_alpha,
                    reg_lambda=reg_lambda,
                    objective="reg:squarederror",
                    random_state=random_state,
                    n_jobs=n_jobs,
                    verbosity=0
                ))
            ])

        elif nome_modelo == "LightGBM":
            n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
            learning_rate = trial.suggest_float("learning_rate", 1e-2, 0.2, log=True)
            max_depth = trial.suggest_int("max_depth", 2, 8)
            num_leaves = trial.suggest_int("num_leaves", 15, 80)
            subsample = trial.suggest_float("subsample", 0.6, 1.0)
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)
            reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10, log=True)
            reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", LGBMRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    num_leaves=num_leaves,
                    subsample=subsample,
                    colsample_bytree=colsample_bytree,
                    reg_alpha=reg_alpha,
                    reg_lambda=reg_lambda,
                    random_state=random_state,
                    n_jobs=n_jobs,
                    verbosity=-1
                ))
            ])

        elif nome_modelo == "MLP":
            hidden_layer_sizes = trial.suggest_categorical(
                "hidden_layer_sizes",
                [(64,), (128,), (128, 64), (256, 128)]
            )
            activation = trial.suggest_categorical("activation", ["relu", "tanh"])
            alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
            learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=pca_n)),
                ("model", MLPRegressor(
                    hidden_layer_sizes=hidden_layer_sizes,
                    activation=activation,
                    alpha=alpha,
                    learning_rate_init=learning_rate_init,
                    max_iter=1500,
                    early_stopping=True,
                    random_state=random_state
                ))
            ])

        else:
            raise ValueError(f"Modelo não reconhecido: {nome_modelo}")

        return pipe

    for nome_modelo in nomes_modelos:
        if verbose:
            print(f"Tunando {nome_modelo}...")

        def objective(trial):
            pipe = montar_pipeline(trial, nome_modelo)
            scores = cross_val_score(
                pipe,
                X_train,
                y_train,
                cv=cv,
                scoring="r2",
                n_jobs=n_jobs
            )
            return np.mean(scores)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=bool(verbose))

        best_params = study.best_params

        class FixedTrial:
            def __init__(self, params):
                self.params = params

            def suggest_int(self, name, low, high, step=1):
                return self.params[name]

            def suggest_float(self, name, low, high, log=False):
                return self.params[name]

            def suggest_categorical(self, name, choices):
                return self.params[name]

        best_pipe = montar_pipeline(FixedTrial(best_params), nome_modelo)
        best_pipe.fit(X_train, y_train)

        y_pred = best_pipe.predict(X_test)
        y_pred = np.asarray(y_pred).ravel()

        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        test_mae = mean_absolute_error(y_test, y_pred)

        if "pca" in best_pipe.named_steps:
            pca_step = best_pipe.named_steps["pca"]
            pca_componentes = pca_step.n_components_
            pca_variancia_explicada = float(np.sum(pca_step.explained_variance_ratio_))
        else:
            pca_componentes = None
            pca_variancia_explicada = None

        if "select" in best_pipe.named_steps:
            k_best = best_pipe.named_steps["select"].k
        else:
            k_best = None

        resultados.append({
            "modelo": nome_modelo,
            "target": target_col,
            "n_amostras": len(df),
            "n_features_originais": X.shape[1],
            "cv_best_r2": study.best_value,
            "test_r2": test_r2,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "pca_componentes": pca_componentes,
            "pca_variancia_explicada": pca_variancia_explicada,
            "k_best": k_best,
            "melhores_params": best_params
        })

        best_models[nome_modelo] = best_pipe

    resultados_df = pd.DataFrame(resultados).sort_values(
        by="test_r2",
        ascending=False
    ).reset_index(drop=True)

    split_data = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    return resultados_df, best_models, split_data

#### Bases Gold

In [3]:
# DataFrame pH
df_gold_ph = pd.read_parquet(
    "s3://projeto-edb-015/gold/afsis/ph/gold_ph_model_ready.parquet",
    storage_options={"anon": False}
)

# DataFrame %C
df_gold_carbono = pd.read_parquet(
    "s3://projeto-edb-015/gold/afsis/carbono/gold_carbono_model_ready.parquet",
    storage_options={"anon": False}
)

# DataFrame P
df_gold_fosforo = pd.read_parquet(
    "s3://projeto-edb-015/gold/afsis/p/gold_p_model_ready.parquet",
    storage_options={"anon": False}
)

# DataFrame ECEC cmolc/ kg soil
df_gold_ecec = pd.read_parquet(
    "s3://projeto-edb-015/gold/afsis/ecec/gold_ecec_model_ready.parquet",
    storage_options={"anon": False}
)

# DataFrame Exbas
df_gold_exbas = pd.read_parquet(
    "s3://projeto-edb-015/gold/afsis/exbas/gold_exbas_model_ready.parquet",
    storage_options={"anon": False}
)

#### Modelagem - pH

In [4]:
resultados_ph, modelos_ph, split_ph = avaliar_modelos_espectrais_optuna(
    df=df_gold_ph,
    target_col="ph",
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=20,
    n_jobs=-1,
    verbose=1
)

resultados_ph

Tunando Ridge...


Best trial: 11. Best value: 0.774585: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s]


Tunando PLS...


Best trial: 2. Best value: 0.774375: 100%|██████████| 20/20 [00:03<00:00,  5.02it/s]


Tunando ElasticNet...


...

Tunando LightGBM...


Best trial: 18. Best value: 0.722671: 100%|██████████| 20/20 [00:17<00:00,  1.15it/s]


Tunando MLP...


Best trial: 19. Best value: 0.0931205: 100%|██████████| 20/20 [00:12<00:00,  1.54it/s]


,modelo,target,n_amostras,n_features_originais,cv_best_r2,test_r2,test_rmse,test_mae,pca_componentes,pca_variancia_explicada,k_best,melhores_params
0,LightGBM,ph,472,2542,0.722671,0.799520,0.462084,0.355574,35.0,0.997750,NaN,"{'pca_n_components': 35, 'n_estimators': 250, 'learning_rate': 0.10956738425937257, 'max_depth': 3, 'num_leaves': 58, 'subsample': 0.91527908534468, 'colsample_bytree': 0.7332380625351959, 'reg_alpha': 0.0485809864989149, 'reg_lambda': 1.8616496223853627}"
1,Lasso,ph,472,2542,0.770028,0.794419,0.467925,0.370964,56.0,0.998858,NaN,"{'pca_n_components': 56, 'alpha': 0.013006515997896486}"
2,XGBoost,ph,472,2542,0.717370,0.790842,0.471978,0.343526,21.0,0.995265,NaN,"{'pca_n_components': 21, 'n_estimators': 300, 'learning_rate': 0.039618301423126896, 'max_depth': 3, 'subsample': 0.8534872781113066, 'colsample_bytree': 0.9458235609397913, 'reg_alpha': 0.028003718902267785, 'reg_lambda': 0.9337085432385142}"
3,Ridge,ph,472,2542,0.774585,0.775037,0.489486,0.386516,30.0,0.997218,NaN,"{'pca_n_components': 30, 'alpha': 13.018691972155313}"
4,ElasticNet,ph,472,2542,0.774485,0.769081,0.495924,0.391617,33.0,0.997563,NaN,"{'pca_n_components': 33, 'alpha': 0.0009595219758209696, 'l1_ratio': 0.4706258252540231}"
5,PLS,ph,472,2542,0.774375,0.768062,0.497017,0.392389,33.0,0.997563,NaN,"{'pca_n_components': 33, 'n_components': 6}"
6,ElasticNet_SelectKBest,ph,472,2542,0.588867,0.662760,0.599315,0.496887,NaN,NaN,297.0,"{'k_best': 297, 'alpha': 0.010251497965459523, 'l1_ratio': 0.0739962601409976}"
7,MLP,ph,472,2542,0.093121,0.211755,0.916254,0.656920,69.0,0.999184,NaN,"{'pca_n_components': 69, 'hidden_layer_sizes': (256, 128), 'activation': 'relu', 'alpha': 3.464540482918949e-05, 'learning_rate_init': 0.008712445720809106}"


#### Modelagem - Carbono

In [5]:
resultados_carbono, modelos_carbono, split_carbono = avaliar_modelos_espectrais_optuna(
    df=df_gold_carbono,
    target_col="pct_c",
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=20,
    n_jobs=-1,
    verbose=1
)

resultados_carbono

Tunando Ridge...


Best trial: 0. Best value: 0.690788: 100%|██████████| 20/20 [00:04<00:00,  4.75it/s]


Tunando PLS...


Best trial: 15. Best value: 0.690224: 100%|██████████| 20/20 [00:04<00:00,  4.72it/s]


Tunando ElasticNet...


...

Tunando LightGBM...


Best trial: 15. Best value: 0.630487: 100%|██████████| 20/20 [00:14<00:00,  1.39it/s]


Tunando MLP...


Best trial: 16. Best value: 0.687734: 100%|██████████| 20/20 [00:08<00:00,  2.36it/s]


,modelo,target,n_amostras,n_features_originais,cv_best_r2,test_r2,test_rmse,test_mae,pca_componentes,pca_variancia_explicada,k_best,melhores_params
0,MLP,pct_c,488,2542,0.687734,0.725095,0.517995,0.315364,18.0,0.994633,NaN,"{'pca_n_components': 18, 'hidden_layer_sizes': (256, 128), 'activation': 'relu', 'alpha': 0.09520534460383046, 'learning_rate_init': 0.007686352346189078}"
1,ElasticNet,pct_c,488,2542,0.689560,0.644357,0.589172,0.399430,54.0,0.998818,NaN,"{'pca_n_components': 54, 'alpha': 0.0341131161743133, 'l1_ratio': 0.35595910511771917}"
2,Lasso,pct_c,488,2542,0.688149,0.639811,0.592926,0.403569,72.0,0.999256,NaN,"{'pca_n_components': 72, 'alpha': 0.01152806843108044}"
3,LightGBM,pct_c,488,2542,0.630487,0.630739,0.600346,0.402261,22.0,0.996005,NaN,"{'pca_n_components': 22, 'n_estimators': 350, 'learning_rate': 0.062098957598138965, 'max_depth': 2, 'num_leaves': 37, 'subsample': 0.7398673012143882, 'colsample_bytree': 0.9873997984807698, 'reg_alpha': 0.0801482686924794, 'reg_lambda': 0.004007282161596952}"
4,Ridge,pct_c,488,2542,0.690788,0.612170,0.615255,0.413675,62.0,0.999050,NaN,"{'pca_n_components': 62, 'alpha': 8.368907342383011}"
5,ElasticNet_SelectKBest,pct_c,488,2542,0.503798,0.576289,0.643087,0.485636,NaN,NaN,297.0,"{'k_best': 297, 'alpha': 0.0008770194151811547, 'l1_ratio': 0.259886621168969}"
6,PLS,pct_c,488,2542,0.690224,0.571634,0.646610,0.427189,54.0,0.998818,NaN,"{'pca_n_components': 54, 'n_components': 6}"
7,XGBoost,pct_c,488,2542,0.665611,0.457460,0.727697,0.378856,18.0,0.994633,NaN,"{'pca_n_components': 18, 'n_estimators': 250, 'learning_rate': 0.034798026722959036, 'max_depth': 3, 'subsample': 0.8207805983259449, 'colsample_bytree': 0.9995255600066517, 'reg_alpha': 0.08120198980152772, 'reg_lambda': 0.0007351198248530422}"


#### Modelagem - Fósforo

In [6]:
resultados_p, modelos_p, split_p = avaliar_modelos_espectrais_optuna(
    df=df_gold_fosforo,
    target_col="olsen_p_mg_kg",
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=20,
    n_jobs=-1,
    verbose=1
)

resultados_p

Tunando Ridge...


Best trial: 19. Best value: 0.000675451: 100%|██████████| 20/20 [00:04<00:00,  4.80it/s]


Tunando PLS...


Best trial: 19. Best value: -0.0362853: 100%|██████████| 20/20 [00:04<00:00,  4.86it/s]


Tunando ElasticNet...


...

Tunando LightGBM...


Best trial: 15. Best value: -0.0135448: 100%|██████████| 20/20 [00:13<00:00,  1.46it/s]


Tunando MLP...


Best trial: 16. Best value: 0.144072: 100%|██████████| 20/20 [00:08<00:00,  2.25it/s]


,modelo,target,n_amostras,n_features_originais,cv_best_r2,test_r2,test_rmse,test_mae,pca_componentes,pca_variancia_explicada,k_best,melhores_params
0,Lasso,olsen_p_mg_kg,489,2542,0.072905,0.086192,4.359303,2.553223,32.0,0.997520,NaN,"{'pca_n_components': 32, 'alpha': 2.459047700012519}"
1,ElasticNet,olsen_p_mg_kg,489,2542,0.074962,0.081988,4.369318,2.543040,62.0,0.999024,NaN,"{'pca_n_components': 62, 'alpha': 8.919316991397258, 'l1_ratio': 0.20690596050440502}"
2,Ridge,olsen_p_mg_kg,489,2542,0.000675,0.056265,4.430111,2.640585,53.0,0.998756,NaN,"{'pca_n_components': 53, 'alpha': 953.5936914731711}"
3,ElasticNet_SelectKBest,olsen_p_mg_kg,489,2542,0.030202,0.047937,4.449614,2.803992,NaN,NaN,45.0,"{'k_best': 45, 'alpha': 0.03880634530820256, 'l1_ratio': 0.28385048921668943}"
4,XGBoost,olsen_p_mg_kg,489,2542,0.002820,0.037338,4.474314,2.202577,50.0,0.998645,NaN,"{'pca_n_components': 50, 'n_estimators': 350, 'learning_rate': 0.010357803095868813, 'max_depth': 2, 'subsample': 0.6916006567347558, 'colsample_bytree': 0.6325055965929153, 'reg_alpha': 0.00010604544921307496, 'reg_lambda': 9.993591149937103}"
5,PLS,olsen_p_mg_kg,489,2542,-0.036285,0.007919,4.542166,2.686907,10.0,0.983384,NaN,"{'pca_n_components': 10, 'n_components': 4}"
6,LightGBM,olsen_p_mg_kg,489,2542,-0.013545,0.000440,4.559255,2.558932,21.0,0.995570,NaN,"{'pca_n_components': 21, 'n_estimators': 150, 'learning_rate': 0.010011882989988046, 'max_depth': 7, 'num_leaves': 40, 'subsample': 0.8187940055752606, 'colsample_bytree': 0.8821636914305189, 'reg_alpha': 9.100320259736442, 'reg_lambda': 0.04454203239212842}"
7,MLP,olsen_p_mg_kg,489,2542,0.144072,-0.325413,5.250070,2.951257,53.0,0.998756,NaN,"{'pca_n_components': 53, 'hidden_layer_sizes': (256, 128), 'activation': 'relu', 'alpha': 4.474076842850793e-05, 'learning_rate_init': 0.000376585147785444}"


#### Modelagem - ECEC

In [7]:
resultados_ecec, modelos_ecec, split_ecec = avaliar_modelos_espectrais_optuna(
    df=df_gold_ecec,
    target_col="ecec_cmolc__kg_soil",
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=20,
    n_jobs=-1,
    verbose=1
)

resultados_ecec

Tunando Ridge...


Best trial: 17. Best value: 0.76079: 100%|██████████| 20/20 [00:04<00:00,  4.79it/s]


Tunando PLS...


Best trial: 2. Best value: 0.75692: 100%|██████████| 20/20 [00:04<00:00,  4.74it/s]


Tunando ElasticNet...


...

Tunando LightGBM...


Best trial: 16. Best value: 0.721421: 100%|██████████| 20/20 [00:16<00:00,  1.21it/s]


Tunando MLP...


Best trial: 18. Best value: 0.766921: 100%|██████████| 20/20 [00:17<00:00,  1.14it/s]


,modelo,target,n_amostras,n_features_originais,cv_best_r2,test_r2,test_rmse,test_mae,pca_componentes,pca_variancia_explicada,k_best,melhores_params
0,MLP,ecec_cmolc__kg_soil,489,2542,0.766921,0.777617,4.676492,3.732369,52.0,0.998721,NaN,"{'pca_n_components': 52, 'hidden_layer_sizes': (64,), 'activation': 'relu', 'alpha': 0.0003221120573579106, 'learning_rate_init': 0.0011302516488766515}"
1,LightGBM,ecec_cmolc__kg_soil,489,2542,0.721421,0.733676,5.117698,3.840099,20.0,0.995272,NaN,"{'pca_n_components': 20, 'n_estimators': 300, 'learning_rate': 0.05995915996025394, 'max_depth': 3, 'num_leaves': 26, 'subsample': 0.7670104688873415, 'colsample_bytree': 0.6885302705025818, 'reg_alpha': 9.462452148966085, 'reg_lambda': 0.018781842583908127}"
2,XGBoost,ecec_cmolc__kg_soil,489,2542,0.712757,0.666876,5.723643,3.824679,47.0,0.998516,NaN,"{'pca_n_components': 47, 'n_estimators': 400, 'learning_rate': 0.02247299297667298, 'max_depth': 3, 'subsample': 0.757885815125591, 'colsample_bytree': 0.8384536418749832, 'reg_alpha': 0.0011313659871437611, 'reg_lambda': 0.00012740017262577626}"
3,Lasso,ecec_cmolc__kg_soil,489,2542,0.757261,0.660212,5.780608,4.343521,65.0,0.999095,NaN,"{'pca_n_components': 65, 'alpha': 0.07615951375580214}"
4,PLS,ecec_cmolc__kg_soil,489,2542,0.756920,0.657445,5.804100,4.403453,58.0,0.998916,NaN,"{'pca_n_components': 58, 'n_components': 17}"
5,ElasticNet,ecec_cmolc__kg_soil,489,2542,0.761044,0.632691,6.010151,4.582347,49.0,0.998606,NaN,"{'pca_n_components': 49, 'alpha': 0.00199994875128154, 'l1_ratio': 0.05545975478736285}"
6,Ridge,ecec_cmolc__kg_soil,489,2542,0.760790,0.621513,6.100921,4.600263,47.0,0.998516,NaN,"{'pca_n_components': 47, 'alpha': 0.42505627397465395}"
7,ElasticNet_SelectKBest,ecec_cmolc__kg_soil,489,2542,0.558496,0.519584,6.873509,5.499852,NaN,NaN,285.0,"{'k_best': 285, 'alpha': 0.01544088393717339, 'l1_ratio': 0.16583628103536732}"


#### Modelagem - ExBas

In [8]:
resultados_exbas, modelos_exbas, split_exbas = avaliar_modelos_espectrais_optuna(
    df=df_gold_exbas,
    target_col="exbas",
    feature_prefix="wn_",
    test_size=0.2,
    random_state=42,
    cv_splits=5,
    n_trials=20,
    n_jobs=-1,
    verbose=1
)

resultados_exbas

Tunando Ridge...


Best trial: 5. Best value: 0.839499: 100%|██████████| 20/20 [00:09<00:00,  2.22it/s]


Tunando PLS...


Best trial: 15. Best value: 0.8376: 100%|██████████| 20/20 [00:09<00:00,  2.19it/s] 


Tunando ElasticNet...


...

Tunando LightGBM...


Best trial: 10. Best value: 0.848607: 100%|██████████| 20/20 [01:21<00:00,  4.06s/it]


Tunando MLP...


Best trial: 15. Best value: 0.863212: 100%|██████████| 20/20 [01:51<00:00,  5.57s/it]


,modelo,target,n_amostras,n_features_originais,cv_best_r2,test_r2,test_rmse,test_mae,pca_componentes,pca_variancia_explicada,k_best,melhores_params
0,LightGBM,exbas,1884,2542,0.848607,0.824770,8.856475,4.268343,33.0,0.997478,NaN,"{'pca_n_components': 33, 'n_estimators': 500, 'learning_rate': 0.06950037554709732, 'max_depth': 5, 'num_leaves': 80, 'subsample': 0.9666397203685703, 'colsample_bytree': 0.8514022278660525, 'reg_alpha': 0.00015650884447663768, 'reg_lambda': 0.006235244652269064}"
1,MLP,exbas,1884,2542,0.863212,0.823621,8.885445,3.766201,56.0,0.998750,NaN,"{'pca_n_components': 56, 'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'alpha': 4.7038239972548396e-05, 'learning_rate_init': 0.00020594189326842143}"
2,Lasso,exbas,1884,2542,0.839321,0.810221,9.216803,4.221835,78.0,0.999228,NaN,"{'pca_n_components': 78, 'alpha': 0.044715569705509134}"
3,ElasticNet,exbas,1884,2542,0.840258,0.799820,9.466003,4.243677,70.0,0.999095,NaN,"{'pca_n_components': 70, 'alpha': 0.17671193870588284, 'l1_ratio': 0.09440714960475004}"
4,Ridge,exbas,1884,2542,0.839499,0.795536,9.566761,4.297343,47.0,0.998410,NaN,"{'pca_n_components': 47, 'alpha': 222.02271552623887}"
5,PLS,exbas,1884,2542,0.837600,0.795507,9.567436,4.455050,49.0,0.998495,NaN,"{'pca_n_components': 49, 'n_components': 8}"
6,XGBoost,exbas,1884,2542,0.853628,0.795052,9.578075,4.190826,47.0,0.998410,NaN,"{'pca_n_components': 47, 'n_estimators': 300, 'learning_rate': 0.02294974811033428, 'max_depth': 6, 'subsample': 0.7304573258265799, 'colsample_bytree': 0.6254916055968371, 'reg_alpha': 0.030258784598733914, 'reg_lambda': 0.9929321552610605}"
7,ElasticNet_SelectKBest,exbas,1884,2542,0.667301,0.616752,13.097740,7.943139,NaN,NaN,300.0,"{'k_best': 300, 'alpha': 0.001024473457675689, 'l1_ratio': 0.6570158501834392}"


#### Salvando resultados

In [11]:
import boto3
from datetime import datetime
import io

# Cliente S3
s3 = boto3.client("s3")

BUCKET = "projeto-edb-015"
BASE_PREFIX = "results"

# Timestamp
timestamp = datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")


def salvar_csv_s3(df, target_name):
    """
    df: DataFrame (pandas)
    target_name: ex: 'ph', 'carbono', etc
    """

    # Nome do arquivo
    file_name = f"resultados_{target_name}_{timestamp}.csv"

    # Caminho S3
    s3_key = f"{BASE_PREFIX}/{target_name}/{file_name}"

    # Converter DataFrame para CSV em memória
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)

    # Upload
    s3.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=csv_buffer.getvalue().encode("utf-8"),
        ContentType="text/csv"
    )

    print(f"Salvo em: s3://{BUCKET}/{s3_key}")


salvar_csv_s3(resultados_ph, "ph")
salvar_csv_s3(resultados_carbono, "carbono")
salvar_csv_s3(resultados_p, "p")
salvar_csv_s3(resultados_ecec, "ecec")
salvar_csv_s3(resultados_exbas, "exbas")

Salvo em: s3://projeto-edb-015/results/ph/resultados_ph_2026-04-20_02-05-55.csv
Salvo em: s3://projeto-edb-015/results/carbono/resultados_carbono_2026-04-20_02-05-55.csv
Salvo em: s3://projeto-edb-015/results/p/resultados_p_2026-04-20_02-05-55.csv
Salvo em: s3://projeto-edb-015/results/ecec/resultados_ecec_2026-04-20_02-05-55.csv
Salvo em: s3://projeto-edb-015/results/exbas/resultados_exbas_2026-04-20_02-05-55.csv
